In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_52.pickle

In [ ]:
%%cudf.pandas.profile
### cell 52 ###

def count_then_return_percent_64(dataframe, x_axis_title):
    # pull the series once and keep operations on GPU
    s = dataframe[x_axis_title]
    counts = s.value_counts(dropna=False)
    total = s.count()
    # vectorized mul/div/round on GPU
    percentages = counts.mul(100).div(total).round(1)
    return percentages

question_of_interest_cell64 = 'What is your current yearly compensation (approximate $USD)?'
# build boolean mask once
mask = responses_df_2022_cell10['In which country do you currently reside?'] == 'India'
India_responses_df_2022_cell64 = responses_df_2022_cell10[mask]
responses_in_order_cell64 = [
    '$0-999', '1,000-1,999', '2,000-2,999', '3,000-3,999',
    '4,000-4,999', '5,000-7,499', '7,500-9,999', '10,000-14,999',
    '15,000-19,999', '20,000-24,999', '25,000-29,999', '30,000-39,999',
    '40,000-49,999', '50,000-59,999', '60,000-69,999', '70,000-79,999',
    '80,000-89,999', '90,000-99,999', '100,000-124,999', '125,000-149,999',
    '150,000-199,999', '200,000-249,999', '250,000-299,999', '300,000-499,999',
    '$500,000-999,999', '>$1,000,000'
]

# compute percentages entirely on GPU and reorder in one step
percentages_cell64 = count_then_return_percent_64(
    India_responses_df_2022_cell64,
    question_of_interest_cell64
).reindex(responses_in_order_cell64)

# inspect result
percentages_cell64.info()